# Comprovació de reproductibilitat

Objectiu: verificar que dos models entrenats amb els **mateixos hiperparàmetres i la mateixa llavor** produeixen exactament els mateixos resultats, i que canviar la llavor genera models **distints**.

S'entrenarà un nombre petit d'epochs per mantenir l'execució ràpida.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
from src.experiment_notebook import setup_experiment, fine_tune_and_evaluate

## Paràmetres comuns

Configuració mínima per fer la prova ràpida: 2 epochs, batch petit.

In [ ]:
COMMON_KWARGS = dict(
    alpha=0.5,
    prev2last_layer_size=16,
    batch_size=32,
    lr=1e-4,
    main_dir='../',
    results_base_dir='../results/reproducibility_check',
)
N_EPOCHS = 2
SEED_A   = 42
SEED_B   = 99  # llavor diferent per mostrar divergència

## Nota sobre `cudnn.benchmark`

`setup_experiment` activa `torch.backends.cudnn.benchmark = True` per velocitat, però això permet que cuDNN esculli algorismes no deterministes.  
Per garantir reproductibilitat total forcem `benchmark=False` i `deterministic=True` just després.

## Entrenament dels tres models

In [ ]:
print("=== Model 1 (seed=42, run A) ===")
ctx1 = setup_experiment(seed=SEED_A, run_name='repro_seed42_A', **COMMON_KWARGS)
torch.backends.cudnn.benchmark    = False   # assegurem determinisme
torch.backends.cudnn.deterministic = True
res1 = fine_tune_and_evaluate(ctx1, epochs=N_EPOCHS)

In [ ]:
print("=== Model 2 (seed=42, run B) ===")
ctx2 = setup_experiment(seed=SEED_A, run_name='repro_seed42_B', **COMMON_KWARGS)
torch.backends.cudnn.benchmark    = False
torch.backends.cudnn.deterministic = True
res2 = fine_tune_and_evaluate(ctx2, epochs=N_EPOCHS)

In [ ]:
print(f"=== Model 3 (seed={SEED_B}, run C) ===")
ctx3 = setup_experiment(seed=SEED_B, run_name=f'repro_seed{SEED_B}_C', **COMMON_KWARGS)
torch.backends.cudnn.benchmark    = False
torch.backends.cudnn.deterministic = True
res3 = fine_tune_and_evaluate(ctx3, epochs=N_EPOCHS)

## Comparació de pesos finals

In [ ]:
def params_as_vector(model):
    """Concatena tots els pesos entrenables en un vector 1-D."""
    return torch.cat([p.detach().cpu().flatten() for p in model.parameters() if p.requires_grad])

w1 = params_as_vector(ctx1['model'])
w2 = params_as_vector(ctx2['model'])
w3 = params_as_vector(ctx3['model'])

diff_same_seed  = (w1 - w2).abs().max().item()
diff_diff_seed  = (w1 - w3).abs().max().item()

print(f"Diferència màx. de pesos (mateixa llavor, 1 vs 2): {diff_same_seed:.2e}")
print(f"Diferència màx. de pesos (llavor diferent, 1 vs 3): {diff_diff_seed:.2e}")

assert diff_same_seed == 0.0, "ERROR: els models amb la mateixa llavor haurien de ser idèntics!"
assert diff_diff_seed  > 0.0, "ERROR: els models amb llavors distintes haurien de diferir!"

print("\n✓ Models 1 i 2 (seed=42):  IDÈNTICS")
print(f"✓ Models 1 i 3 (seed 42 vs {SEED_B}): DISTINTS")

## Comparació de prediccions al conjunt de test

In [ ]:
import torch.nn.functional as F

def get_probabilities(ctx):
    model  = ctx['model']
    loader = ctx['test_loader']
    device = ctx['device']
    model.eval()
    probs_list = []
    with torch.no_grad():
        for images, labels, genders in loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)[:, 1].cpu()
            probs_list.append(probs)
    return torch.cat(probs_list)

p1 = get_probabilities(ctx1)
p2 = get_probabilities(ctx2)
p3 = get_probabilities(ctx3)

pred_diff_same = (p1 - p2).abs().max().item()
pred_diff_diff = (p1 - p3).abs().max().item()

print(f"Diferència màx. de probabilitats (mateixa llavor, 1 vs 2): {pred_diff_same:.2e}")
print(f"Diferència màx. de probabilitats (llavor diferent, 1 vs 3): {pred_diff_diff:.2e}")

assert pred_diff_same == 0.0, "ERROR: les prediccions haurien de ser idèntiques amb la mateixa llavor!"
assert pred_diff_diff  > 0.0, "ERROR: les prediccions haurien de diferir amb llavors distintes!"

print("\n✓ Prediccions 1 i 2 (seed=42):  IDÈNTIQUES")
print(f"✓ Prediccions 1 i 3 (seed 42 vs {SEED_B}): DISTINTES")

## Resum de mètriques

In [ ]:
import pandas as pd

rows = []
for label, res in [(f'Model 1 (seed={SEED_A}, run A)', res1),
                   (f'Model 2 (seed={SEED_A}, run B)', res2),
                   (f'Model 3 (seed={SEED_B}, run C)', res3)]:
    rows.append({
        'Model': label,
        'AUC global':   round(res.get('auc_global', float('nan')), 4),
        'ACC global':   round(res.get('acc_global', float('nan')), 4),
        'AUC M':        round(res.get('auc_male',   float('nan')), 4),
        'AUC F':        round(res.get('auc_female', float('nan')), 4),
    })

df_res = pd.DataFrame(rows).set_index('Model')
display(df_res)

print("\nConclusions:")
print(f"  Models 1 i 2 haurien de tenir exactament les mateixes mètriques (mateixa llavor).")
print(f"  El model 3 (seed={SEED_B}) pot tenir mètriques lleugerament distintes.")

## Conclusions

| Condició | Resultat esperat |
|---|---|
| Mateixa llavor, mateixos hiperparàmetres | Pesos idèntics, prediccions idèntiques, mètriques idèntiques |
| Llavor diferent, mateixos hiperparàmetres | Pesos distints, prediccions distintes, mètriques lleugerament distintes |

**Per tant**, la variabilitat observada al sweep és exclusivament deguda a les **llavors distintes** (`BASE_SEED`, `BASE_SEED+1`, `BASE_SEED+2`), no a cap bug ni comportament no desitjat.

> **Nota**: Per garantir reproductibilitat total cal `torch.backends.cudnn.benchmark = False` i `deterministic = True`. Amb `benchmark=True` (per defecte a `setup_experiment`) pot haver-hi petites diferències numèriques en GPU degudes als algorismes cuDNN no deterministes.